In [2]:
import pandas as pd
import os

# --- CẤU HÌNH ĐƯỜNG DẪN FILE ---
files = {
    'Metadata': 'linkedin_job_postings.csv',
    'Summary': 'job_summary.csv',
    'Skills': 'job_skills.csv'
}

def get_file_size_mb(path):
    """Hàm lấy dung lượng file (MB)"""
    if os.path.exists(path):
        return os.path.getsize(path) / (1024 * 1024)
    return 0

def analyze_large_file(file_path, name):
    print(f"============== PHÂN TÍCH FILE: {name} ==============")
    
    # 1. Kiểm tra dung lượng
    size_mb = get_file_size_mb(file_path)
    print(f"📦 Dung lượng trên ổ cứng: {size_mb:.2f} MB")
    
    if size_mb == 0:
        print("❌ Không tìm thấy file!")
        return

    # 2. Đếm số dòng (Dùng chunk để không nổ RAM với file 5GB)
    total_rows = 0
    missing_ids = 0
    duplicate_ids = 0
    
    # Chỉ đọc cột job_link để check ID (nhẹ nhất có thể)
    # Với file summary/skills, ta cần check xem có bao nhiêu job có nội dung
    try:
        # Dùng set để đếm unique ID
        unique_ids = set()
        
        # Đọc từng miếng 100k dòng
        chunk_size = 100_000
        for chunk in pd.read_csv(file_path, usecols=['job_link'], chunksize=chunk_size):
            # Cập nhật tổng số dòng
            rows_in_chunk = len(chunk)
            total_rows += rows_in_chunk
            
            # Đếm null ID
            missing_ids += chunk['job_link'].isnull().sum()
            
            # Cập nhật unique ID
            unique_ids.update(chunk['job_link'].dropna())
            
        print(f"🔢 Tổng số dòng (Rows): {total_rows:,.0f}")
        print(f"🆔 Số lượng Job duy nhất (Unique IDs): {len(unique_ids):,.0f}")
        
        # Tính toán trùng lặp
        duplicates = total_rows - len(unique_ids) - missing_ids
        print(f"⚠️  Số dòng bị trùng lặp ID (Duplicates): {duplicates:,.0f}")
        print(f"❓  Số dòng bị thiếu ID (Null): {missing_ids:,.0f}")
        
    except Exception as e:
        print(f"❌ Lỗi khi đọc file: {e}")
    print("\n")

# --- CHẠY PHÂN TÍCH ---
print("🚀 BẮT ĐẦU QUÉT DỮ LIỆU...\n")

# 1. Phân tích file Metadata (Quan trọng nhất)
# File này chứa thông tin các cột khác nên ta đọc thử header để xem tên cột
df_head = pd.read_csv(files['Metadata'], nrows=0)
print(f"📋 Các cột trong Metadata: {list(df_head.columns)}")
analyze_large_file(files['Metadata'], "METADATA (Postings)")

# 2. Phân tích file Summary (Nặng nhất)
analyze_large_file(files['Summary'], "CONTENT (Job Summary)")

# 3. Phân tích file Skills
analyze_large_file(files['Skills'], "SKILLS (Job Skills)")

🚀 BẮT ĐẦU QUÉT DỮ LIỆU...

📋 Các cột trong Metadata: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']
============== PHÂN TÍCH FILE: METADATA (Postings) ==============
📦 Dung lượng trên ổ cứng: 396.09 MB
🔢 Tổng số dòng (Rows): 1,348,454
🆔 Số lượng Job duy nhất (Unique IDs): 1,348,454
⚠️  Số dòng bị trùng lặp ID (Duplicates): 0
❓  Số dòng bị thiếu ID (Null): 0


============== PHÂN TÍCH FILE: CONTENT (Job Summary) ==============
📦 Dung lượng trên ổ cứng: 4865.66 MB
🔢 Tổng số dòng (Rows): 1,297,332
🆔 Số lượng Job duy nhất (Unique IDs): 1,297,332
⚠️  Số dòng bị trùng lặp ID (Duplicates): 0
❓  Số dòng bị thiếu ID (Null): 0


============== PHÂN TÍCH FILE: SKILLS (Job Skills) ==============
📦 Dung lượng trên ổ cứng: 641.55 MB
🔢 Tổng số dòng (Rows): 1,296,381
🆔 Số lượng Job duy nhất (Unique IDs): 1,296,381
⚠️  Số dòng bị trùng lặp 

Làm sạch trước khi hợp nhất 3 file

In [3]:
import pandas as pd
import re
import os

# --- 1. CẤU HÌNH & TỪ KHÓA ---
# Tên file đầu vào
file_raw_postings = 'linkedin_job_postings.csv'
file_raw_summary = 'job_summary.csv'
file_raw_skills = 'job_skills.csv'

# Tên file đầu ra (Đã làm sạch)
file_clean_postings = 'clean_postings.csv'
file_clean_summary = 'clean_summary.csv'
file_clean_skills = 'clean_skills.csv'

# Từ khóa định nghĩa "Việc làm IT" (Để lọc rác ngành khác)
it_keywords = [
    'software', 'developer', 'engineer', 'data', 'analyst', 'scientist',
    'it', 'tech', 'system', 'network', 'cloud', 'security', 'full stack',
    'backend', 'frontend', 'devops', 'machine learning', 'ai', 'programmer',
    'database', 'web', 'ios', 'android', 'cyber', 'linux', 'java', 'python'
]

# --- 2. CÁC HÀM XỬ LÝ (HELPER FUNCTIONS) ---

def clean_text_content(text):
    """
    Hàm làm sạch nội dung văn bản (theo slide: sửa chữa điểm khuyết):
    1. Xóa HTML tags (<br>, <p>...)
    2. Xóa URL rác
    3. Xóa khoảng trắng thừa
    """
    if not isinstance(text, str):
        return ""
    
    # Xóa HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Xóa URLs (http/https)
    text = re.sub(r'http\S+', '', text)
    # Thay thế nhiều khoảng trắng thành 1 khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_it_job_title(title):
    """Kiểm tra xem Title có chứa từ khóa IT không"""
    if not isinstance(title, str): return False
    return any(keyword in title.lower() for keyword in it_keywords)

# --- 3. BẮT ĐẦU QUY TRÌNH CLEANING ---
print("🚀 BẮT ĐẦU CLEANING PIPELINE...\n")

# ==========================================
# BƯỚC 1: XỬ LÝ FILE METADATA (Xương sống)
# ==========================================
print(f"1️⃣  Đang xử lý {file_raw_postings}...")

# Đọc file (chỉ lấy các cột quan trọng để nhẹ máy)
cols_needed = ['job_link', 'job_title', 'company', 'job_location', 'job_level', 'first_seen']
df_postings = pd.read_csv(file_raw_postings, usecols=cols_needed)

print(f"   - Tổng số dòng ban đầu: {len(df_postings):,}")

# 1.1 Loại bỏ dòng thiếu dữ liệu quan trọng (Missing Values)
df_postings.dropna(subset=['job_title', 'job_link'], inplace=True)

# 1.2 Loại bỏ trùng lặp (Duplicates) - Giữ dòng mới nhất
df_postings.drop_duplicates(subset=['job_link'], keep='last', inplace=True)

# 1.3 Lọc ngành IT (Domain filtering)
df_postings = df_postings[df_postings['job_title'].apply(is_it_job_title)]

print(f"   - Số lượng Job IT sạch còn lại: {len(df_postings):,}")

# Lưu file sạch ra đĩa
df_postings.to_csv(file_clean_postings, index=False)
print(f"   -> Đã lưu: {file_clean_postings}")

# Tạo tập hợp ID hợp lệ để lọc 2 file kia (Consistency Check)
valid_job_ids = set(df_postings['job_link'])

# Dọn dẹp RAM
del df_postings 

# ==========================================
# BƯỚC 2: XỬ LÝ FILE SUMMARY (Nặng nhất)
# ==========================================
print(f"\n2️⃣  Đang xử lý {file_raw_summary} (Chế độ Chunking)...")

# Xóa file cũ nếu tồn tại để ghi mới
if os.path.exists(file_clean_summary):
    os.remove(file_clean_summary)

chunk_size = 50000 # Đọc mỗi lần 50k dòng
processed_count = 0

# Vòng lặp đọc và xử lý từng phần
for i, chunk in enumerate(pd.read_csv(file_raw_summary, chunksize=chunk_size)):
    # 2.1 Lọc: Chỉ giữ lại các dòng có ID nằm trong danh sách IT hợp lệ
    chunk_filtered = chunk[chunk['job_link'].isin(valid_job_ids)].copy()
    
    if not chunk_filtered.empty:
        # 2.2 Sửa chữa: Làm sạch text (xóa HTML)
        chunk_filtered['job_summary'] = chunk_filtered['job_summary'].apply(clean_text_content)
        
        # 2.3 Loại bỏ dòng rỗng sau khi clean
        chunk_filtered = chunk_filtered[chunk_filtered['job_summary'] != ""]
        
        # Lưu vào file (Append mode)
        write_header = (i == 0) # Chỉ ghi header lần đầu
        chunk_filtered.to_csv(file_clean_summary, mode='a', index=False, header=write_header)
        processed_count += len(chunk_filtered)
        
    print(f"   -> Đã xử lý xong chunk {i+1}...", end='\r')

print(f"\n   -> Hoàn tất! Tổng số Summary sạch: {processed_count:,}")
print(f"   -> Đã lưu: {file_clean_summary}")

# ==========================================
# BƯỚC 3: XỬ LÝ FILE SKILLS
# ==========================================
print(f"\n3️⃣  Đang xử lý {file_raw_skills}...")

df_skills = pd.read_csv(file_raw_skills)

# 3.1 Đồng bộ dữ liệu: Chỉ giữ skill của job IT hợp lệ
df_skills = df_skills[df_skills['job_link'].isin(valid_job_ids)]

# 3.2 Chuẩn hóa: Chuyển về chữ thường
df_skills['job_skills'] = df_skills['job_skills'].str.lower().str.strip()

# Lưu file
df_skills.to_csv(file_clean_skills, index=False)
print(f"   -> Đã lưu: {file_clean_skills}")
print(f"   - Số lượng Skills sạch: {len(df_skills):,}")

print("\n✅ HOÀN THÀNH BƯỚC CLEANING!")

🚀 BẮT ĐẦU CLEANING PIPELINE...

1️⃣  Đang xử lý linkedin_job_postings.csv...
   - Tổng số dòng ban đầu: 1,348,454
   - Số lượng Job IT sạch còn lại: 503,623
   -> Đã lưu: clean_postings.csv

2️⃣  Đang xử lý job_summary.csv (Chế độ Chunking)...
   -> Đã xử lý xong chunk 26...
   -> Hoàn tất! Tổng số Summary sạch: 483,373
   -> Đã lưu: clean_summary.csv

3️⃣  Đang xử lý job_skills.csv...
   -> Đã lưu: clean_skills.csv
   - Số lượng Skills sạch: 483,043

✅ HOÀN THÀNH BƯỚC CLEANING!


Hợp Nhất 3 file

In [4]:
import pandas as pd

# --- CẤU HÌNH ---
file_clean_postings = 'clean_postings.csv'
file_clean_summary = 'clean_summary.csv'
file_clean_skills = 'clean_skills.csv'
output_file = 'merged_it_jobs_dataset.csv'

print("🚀 BẮT ĐẦU QUÁ TRÌNH INTEGRATION (HỢP NHẤT)...\n")

# ==========================================
# BƯỚC 1: LOAD DỮ LIỆU ĐÃ LÀM SẠCH
# ==========================================
print("1️⃣  Đang tải các file dữ liệu sạch lên RAM...")

# Đọc file Postings (Bảng chính - Master Table)
df_main = pd.read_csv(file_clean_postings)
print(f"   - Postings (Gốc): {len(df_main):,} dòng")

# Đọc file Summary (Bảng phụ 1)
df_summary = pd.read_csv(file_clean_summary)
print(f"   - Summary: {len(df_summary):,} dòng")

# Đọc file Skills (Bảng phụ 2)
df_skills = pd.read_csv(file_clean_skills)
print(f"   - Skills: {len(df_skills):,} dòng")

# ==========================================
# BƯỚC 2 & 3: THỰC THI LEFT JOIN
# ==========================================
print("\n2️⃣  Đang thực hiện ghép nối (Left Join)...")

# Lần 1: Ghép Postings + Summary
# on='job_link': Dựa vào cột job_link để ghép
# how='left': Giữ nguyên tất cả dòng của df_main, nếu summary thiếu thì để trống (NaN)
merged_df = pd.merge(df_main, df_summary, on='job_link', how='left')

# Lần 2: Ghép kết quả + Skills
merged_df = pd.merge(merged_df, df_skills, on='job_link', how='left')

# ==========================================
# BƯỚC 4: KIỂM TRA TOÀN VẸN (INTEGRITY CHECK)
# ==========================================
print("\n3️⃣  Kiểm tra dữ liệu sau hợp nhất...")

initial_count = len(df_main)
final_count = len(merged_df)

print(f"   - Số dòng ban đầu: {initial_count:,}")
print(f"   - Số dòng sau gộp: {final_count:,}")

if initial_count == final_count:
    print("   ✅ Kiểm tra thành công: Không bị nhân bản dữ liệu.")
else:
    print("   ⚠️ Cảnh báo: Số dòng đã thay đổi! (Có thể do trùng lặp ID trong file Skills hoặc Summary)")
    # Xử lý nhanh nếu bị trùng: Gom nhóm lại
    # (Code này phòng hờ: Nếu 1 job có 2 dòng skill tách biệt, nó sẽ gộp lại thành 1 dòng)
    if final_count > initial_count:
        print("   -> Đang tự động xử lý trùng lặp...")
        merged_df.drop_duplicates(subset=['job_link'], keep='first', inplace=True)
        print(f"   -> Số dòng sau khi fix: {len(merged_df):,}")

# Kiểm tra dữ liệu thiếu sau khi gộp
missing_summary = merged_df['job_summary'].isnull().sum()
missing_skills = merged_df['job_skills'].isnull().sum()
print(f"   - Số job thiếu mô tả (Summary): {missing_summary:,}")
print(f"   - Số job thiếu kỹ năng (Skills): {missing_skills:,}")

# ==========================================
# LƯU KẾT QUẢ
# ==========================================
print(f"\n💾 Đang lưu file hợp nhất: {output_file}")
merged_df.to_csv(output_file, index=False)

print("\n✅ HOÀN TẤT MODULE INTEGRATION!")
print(f"File '{output_file}' đã sẵn sàng cho bước Transformation & Encoding.")

🚀 BẮT ĐẦU QUÁ TRÌNH INTEGRATION (HỢP NHẤT)...

1️⃣  Đang tải các file dữ liệu sạch lên RAM...
   - Postings (Gốc): 503,623 dòng
   - Summary: 483,373 dòng
   - Skills: 483,043 dòng

2️⃣  Đang thực hiện ghép nối (Left Join)...

3️⃣  Kiểm tra dữ liệu sau hợp nhất...
   - Số dòng ban đầu: 503,623
   - Số dòng sau gộp: 503,623
   ✅ Kiểm tra thành công: Không bị nhân bản dữ liệu.
   - Số job thiếu mô tả (Summary): 20,250
   - Số job thiếu kỹ năng (Skills): 21,249

💾 Đang lưu file hợp nhất: merged_it_jobs_dataset.csv

✅ HOÀN TẤT MODULE INTEGRATION!
File 'merged_it_jobs_dataset.csv' đã sẵn sàng cho bước Transformation & Encoding.


Kiểm tra sau khi gộp

In [7]:
import pandas as pd

# Đường dẫn file
file_path = 'merged_it_jobs_dataset.csv'

print(f"⏳ Đang đọc file '{file_path}'...")
df = pd.read_csv(file_path)

# --- HÀM HIỂN THỊ BẢNG ĐẸP ---
def print_table(dataframe, title):
    print(f"\n🔹 {title}")
    try:
        # to_markdown giúp kẻ bảng đẹp mắt
        print(dataframe.to_markdown(index=False, numalign="left", stralign="left"))
    except ImportError:
        # Fallback nếu máy bạn chưa cài tabulate
        print("⚠️ (Bạn nên cài 'pip install tabulate' để xem bảng đẹp hơn)")
        print(dataframe.to_string(index=False))

# ======================================================
# 1. HIỂN THỊ 5 DÒNG ĐẦU TIÊN (DẠNG BẢNG)
# ======================================================
# Chỉ chọn các cột quan trọng để bảng không bị vỡ giao diện
preview_cols = ['job_title', 'company', 'job_location', 'job_skills']

# Tạo cột mô tả ngắn gọn (chỉ lấy 50 ký tự đầu) để xem trước
if 'job_summary' in df.columns:
    df['summary_preview'] = df['job_summary'].astype(str).str[:50] + "..."
    preview_cols.append('summary_preview')

# In bảng dữ liệu mẫu
print_table(df[preview_cols].head(5), "1. MẪU DỮ LIỆU (TOP 5 ROWs)")

# ======================================================
# 2. THỐNG KÊ DỮ LIỆU THIẾU (DẠNG BẢNG)
# ======================================================
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

stat_df = pd.DataFrame({
    'Tên cột': df.columns,
    'Số dòng thiếu': missing_count,
    'Tỷ lệ thiếu (%)': missing_percent.map('{:.2f}%'.format),
    'Kiểu dữ liệu': df.dtypes.astype(str)
})

# Sắp xếp giảm dần theo số lượng thiếu
stat_df = stat_df.sort_values(by='Số dòng thiếu', ascending=False)

# In bảng thống kê
print_table(stat_df, "2. THỐNG KÊ DỮ LIỆU THIẾU (MISSING VALUES)")

print(f"\n✅ Tổng số dòng dữ liệu: {len(df):,} dòng")

⏳ Đang đọc file 'merged_it_jobs_dataset.csv'...

🔹 1. MẪU DỮ LIỆU (TOP 5 ROWs)
| job_title                                           | company                      | job_location               | job_skills                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  | summary_preview        

Loại bỏ các dòng dữ liệu có job_skill hoặc job_summary(chi tiết công việc bị trống)

In [8]:
import pandas as pd
import os

# --- CẤU HÌNH ---
input_file = 'merged_it_jobs_dataset.csv'
output_file = 'final_clean_dataset.csv'

print(f"🚀 Đang tải file '{input_file}' để xử lý lần cuối...")

if os.path.exists(input_file):
    df = pd.read_csv(input_file)
    
    # 1. Thống kê trước khi xóa
    total_before = len(df)
    print(f"📊 Tổng số dòng ban đầu: {total_before:,}")
    print("\n🔍 Chi tiết số lượng Null từng cột:")
    print(df.isnull().sum())

    # 2. Hành động: XÓA SẠCH (Drop NA)
    # how='any': Chỉ cần 1 cột bị null là xóa cả dòng đó ngay lập tức
    df_clean = df.dropna(how='any')
    
    # 3. Thống kê sau khi xóa
    total_after = len(df_clean)
    removed_rows = total_before - total_after
    
    print("-" * 50)
    print(f"🗑️ ĐÃ XÓA: {removed_rows:,} dòng (Do thiếu dữ liệu)")
    print(f"✅ CÒN LẠI: {total_after:,} dòng (Dữ liệu sạch 100%)")
    print("-" * 50)
    
    # 4. Lưu file cuối cùng
    df_clean.to_csv(output_file, index=False)
    print(f"💾 Đã lưu kết quả vào file mới: '{output_file}'")
    
    # Hiển thị mẫu để kiểm chứng
    print("\n👁️  Mẫu dữ liệu sạch (5 dòng đầu):")
    # Hiển thị ngắn gọn để dễ nhìn
    preview_cols = ['job_title', 'company', 'job_skills']
    print(df_clean[preview_cols].head().to_markdown(index=False, numalign="left", stralign="left"))
    
else:
    print(f"❌ Lỗi: Không tìm thấy file '{input_file}'. Hãy chắc chắn bạn đã chạy bước Hợp nhất (Merge) trước đó.")

🚀 Đang tải file 'merged_it_jobs_dataset.csv' để xử lý lần cuối...
📊 Tổng số dòng ban đầu: 503,623

🔍 Chi tiết số lượng Null từng cột:
job_link            0
job_title           0
company             5
job_location       12
first_seen          0
job_level           0
job_summary     20250
job_skills      21249
dtype: int64
--------------------------------------------------
🗑️ ĐÃ XÓA: 21,265 dòng (Do thiếu dữ liệu)
✅ CÒN LẠI: 482,358 dòng (Dữ liệu sạch 100%)
--------------------------------------------------
💾 Đã lưu kết quả vào file mới: 'final_clean_dataset.csv'

👁️  Mẫu dữ liệu sạch (5 dòng đầu):
| job_title                                               | company                     | job_skills                                                                                                                                                                                                                                                                                                        

LOẠI BỎ CÁC CỘT KHÔNG CẦN THIẾT, CHỈ GIỮ LẠI CÁC CỘT QUAN TRỌNG, CHUẨN HÓA DATE

In [1]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv("job_2025.csv")

# Chỉ giữ lại các cột cần thiết
df_clean = df[["jobtitle", "skills", "jobdescription"]].copy()

# Đổi tên cột cho đúng yêu cầu
df_clean = df_clean.rename(columns={
    "jobtitle": "title",
    "jobdescription": "summary",
    "skills": "skill"
})

# Thêm cột năm
df_clean["year"] = 2025

# Lưu lại file sau khi clean bước 1
df_clean.to_csv("job_2025_step1_clean.csv", index=False)

print(df_clean.head())


                                               title  \
0                           AUTOMATION TEST ENGINEER   
1                      Information Security Engineer   
2                       Business Solutions Architect   
3  Java Developer (mid level)- FT- GREAT culture,...   
4                                    DevOps Engineer   

                                               skill  \
0                                          SEE BELOW   
1  linux/unix, network monitoring, incident respo...   
2  Enterprise Solutions Architecture, business in...   
3                         Please see job description   
4  Configuration Management, Developer, Linux, Ma...   

                                             summary  year  
0  Looking for Selenium engineers...must have sol...  2025  
1  The University of Chicago has a rapidly growin...  2025  
2  GalaxE.SolutionsEvery day, our solutions affec...  2025  
3  Java DeveloperFull-time/direct-hireBolingbrook...  2025  
4  Midtown based high

In [2]:
import re
import pandas as pd

TARGET_LANGS = {
    "python", "c", "c++", "java", "c#", "javascript", "go", "rust", "typescript", "php", "kotlin"
}

# Map thư viện/framework/tool phổ biến -> ngôn ngữ
# (bạn có thể bổ sung thêm theo data thực tế của bạn)
LIB_TO_LANG = {
    # Python
    "numpy": "python", "pandas": "python", "scipy": "python", "sklearn": "python", "scikit-learn": "python",
    "tensorflow": "python", "pytorch": "python", "keras": "python", "django": "python", "flask": "python",
    "fastapi": "python", "pytest": "python",

    # Java
    "spring": "java", "spring boot": "java", "hibernate": "java", "maven": "java", "gradle": "java",

    # JavaScript
    "node": "javascript", "nodejs": "javascript", "node.js": "javascript",
    "react": "javascript", "reactjs": "javascript",
    "vue": "javascript", "vuejs": "javascript",
    "angular": "javascript",
    "express": "javascript", "next": "javascript", "nextjs": "javascript", "next.js": "javascript",

    # TypeScript
    "nestjs": "typescript", "nest": "typescript",

    # C#
    ".net": "c#", "dotnet": "c#", "asp.net": "c#", "aspnet": "c#", "entity framework": "c#",

    # PHP
    "laravel": "php", "symfony": "php", "wordpress": "php",

    # Go
    "golang": "go", "gin": "go", "echo": "go", "fiber": "go",

    # Rust
    "actix": "rust", "actix-web": "rust", "rocket": "rust", "tokio": "rust",

    # Kotlin
    "ktor": "kotlin",
}

# Chuẩn hóa trực tiếp các cách viết ngôn ngữ hay gặp
CANON = {
    "py": "python",
    "c sharp": "c#",
    "csharp": "c#",
    "js": "javascript",
    "ts": "typescript",
    "golang": "go",  # cũng có trong LIB_TO_LANG nhưng để an toàn
}

# Regex bắt các token C++ / C# / TypeScript / JavaScript ... kể cả khi lẫn trong text
LANG_PATTERNS = [
    (re.compile(r"^\s*c\+\+\s*$", re.I), "c++"),
    (re.compile(r"^\s*c#\s*$", re.I), "c#"),
    (re.compile(r"^\s*javascript\s*$", re.I), "javascript"),
    (re.compile(r"^\s*typescript\s*$", re.I), "typescript"),
    (re.compile(r"^\s*python\s*$", re.I), "python"),
    (re.compile(r"^\s*java\s*$", re.I), "java"),
    (re.compile(r"^\s*go\s*$", re.I), "go"),
    (re.compile(r"^\s*rust\s*$", re.I), "rust"),
    (re.compile(r"^\s*php\s*$", re.I), "php"),
    (re.compile(r"^\s*kotlin\s*$", re.I), "kotlin"),
    # "C" để cuối, tránh ăn nhầm "C++" / "C#"
    (re.compile(r"^\s*c\s*$", re.I), "c"),
]

SPLIT_RE = re.compile(r"[,\|;/\n]+")  # tách theo nhiều kiểu phân cách

def normalize_one_skill(token: str) -> str | None:
    if token is None:
        return None

    s = token.strip().lower()
    if not s:
        return None

    # bỏ các ký tự lặt vặt ở 2 đầu
    s = re.sub(r"^[\-\*\•\.\(\)\[\]\{\}\s]+|[\-\*\•\.\(\)\[\]\{\}\s]+$", "", s).strip()
    if not s:
        return None

    # canonical aliases (js->javascript, csharp->c#, ts->typescript...)
    s = CANON.get(s, s)

    # thư viện/framework -> map ngôn ngữ
    if s in LIB_TO_LANG:
        s = LIB_TO_LANG[s]

    # match trực tiếp ngôn ngữ
    for pat, lang in LANG_PATTERNS:
        if pat.match(s):
            return lang  # đã là lowercase

    # một số skill có thể viết kiểu "Java (Spring)" hoặc "Python/Django"
    # thử tìm xem trong chuỗi có chứa tên thư viện/ngôn ngữ
    # 1) tìm theo LIB_TO_LANG
    for lib, lang in LIB_TO_LANG.items():
        if lib in s:
            return lang

    # 2) tìm theo keywords ngôn ngữ
    for lang in TARGET_LANGS:
        if lang in s:
            # tránh trường hợp "c" nằm trong nhiều từ khác: chỉ nhận "c" nếu là từ riêng
            if lang == "c":
                if re.search(r"(^|\W)c(\W|$)", s):
                    return "c"
            else:
                return lang

    return None

def clean_skill_cell(cell) -> str:
    if pd.isna(cell):
        return ""

    raw = str(cell)
    parts = [p.strip() for p in SPLIT_RE.split(raw) if p.strip()]

    normalized = []
    seen = set()
    for p in parts:
        lang = normalize_one_skill(p)
        if lang and lang in TARGET_LANGS and lang not in seen:
            seen.add(lang)
            normalized.append(lang)

    # yêu cầu: cuối cùng chuẩn hóa chữ thường hết -> đã lowercase
    return ",".join(normalized)

# ====== APPLY ======
# df_clean là dataframe sau bước 1 (có cột: title, summary, skill, year)
df_clean["skill"] = df_clean["skill"].apply(clean_skill_cell)


CHUẨN HÓA SKILL
1. LOẠI BỎ CÁC SKILL KHÔNG PHẢI LÀ NGÔN NGỮ LẬP TRÌNH
2. ĐƯA SKILL LÀ THƯ VIỆN VỀ DẠNG NGÔN NGỮ LẬP TRÌNH
3. ĐƯA VỀ CHỮ THƯỜNG

In [3]:
import re
import pandas as pd

# ====== CONFIG ======
INPUT_PATH = "job_2025_step1_clean.csv"
OUTPUT_PATH = "job_2025_step2_skill_clean.csv"

TARGET_LANGS = {
    "python", "c", "c++", "java", "c#", "javascript", "go", "rust", "typescript", "php", "kotlin"
}

# thư viện/framework/tool -> ngôn ngữ
LIB_TO_LANG = {
    # Python
    "numpy": "python", "pandas": "python", "scipy": "python", "sklearn": "python", "scikit-learn": "python",
    "tensorflow": "python", "pytorch": "python", "keras": "python", "django": "python", "flask": "python",
    "fastapi": "python", "pytest": "python",

    # Java
    "spring": "java", "spring boot": "java", "hibernate": "java", "maven": "java", "gradle": "java",

    # JavaScript
    "node": "javascript", "nodejs": "javascript", "node.js": "javascript",
    "react": "javascript", "reactjs": "javascript",
    "vue": "javascript", "vuejs": "javascript",
    "angular": "javascript",
    "express": "javascript", "next": "javascript", "nextjs": "javascript", "next.js": "javascript",

    # TypeScript
    "nestjs": "typescript", "nest": "typescript",

    # C#
    ".net": "c#", "dotnet": "c#", "asp.net": "c#", "aspnet": "c#", "entity framework": "c#",

    # PHP
    "laravel": "php", "symfony": "php", "wordpress": "php",

    # Go
    "golang": "go", "gin": "go", "echo": "go", "fiber": "go",

    # Rust
    "actix": "rust", "actix-web": "rust", "rocket": "rust", "tokio": "rust",

    # Kotlin
    "ktor": "kotlin",
}

# alias ngôn ngữ
CANON = {
    "py": "python",
    "c sharp": "c#",
    "csharp": "c#",
    "js": "javascript",
    "ts": "typescript",
    "golang": "go",
}

# Ưu tiên match C++/C# trước, C để cuối tránh ăn nhầm
LANG_PATTERNS = [
    (re.compile(r"^\s*c\+\+\s*$", re.I), "c++"),
    (re.compile(r"^\s*c#\s*$", re.I), "c#"),
    (re.compile(r"^\s*javascript\s*$", re.I), "javascript"),
    (re.compile(r"^\s*typescript\s*$", re.I), "typescript"),
    (re.compile(r"^\s*python\s*$", re.I), "python"),
    (re.compile(r"^\s*java\s*$", re.I), "java"),
    (re.compile(r"^\s*go\s*$", re.I), "go"),
    (re.compile(r"^\s*rust\s*$", re.I), "rust"),
    (re.compile(r"^\s*php\s*$", re.I), "php"),
    (re.compile(r"^\s*kotlin\s*$", re.I), "kotlin"),
    (re.compile(r"^\s*c\s*$", re.I), "c"),
]

SPLIT_RE = re.compile(r"[,\|;/\n]+")

def normalize_one_skill(token: str):
    if token is None:
        return None

    s = str(token).strip().lower()
    if not s:
        return None

    # dọn ký tự rác 2 đầu
    s = re.sub(r"^[\-\*\•\.\(\)\[\]\{\}\s]+|[\-\*\•\.\(\)\[\]\{\}\s]+$", "", s).strip()
    if not s:
        return None

    # alias
    s = CANON.get(s, s)

    # lib/framework -> lang
    if s in LIB_TO_LANG:
        s = LIB_TO_LANG[s]

    # match trực tiếp
    for pat, lang in LANG_PATTERNS:
        if pat.match(s):
            return lang

    # skill kiểu "Spring (Java)" / "Python/Django" / "Node.js"...
    for lib, lang in LIB_TO_LANG.items():
        if lib in s:
            return lang

    # fallback: chứa keyword ngôn ngữ
    for lang in TARGET_LANGS:
        if lang == "c":
            if re.search(r"(^|\W)c(\W|$)", s):
                return "c"
        else:
            if lang in s:
                return lang

    return None

def clean_skill_cell(cell) -> str:
    if pd.isna(cell):
        return ""

    parts = [p.strip() for p in SPLIT_RE.split(str(cell)) if p.strip()]

    seen = set()
    out = []
    for p in parts:
        lang = normalize_one_skill(p)
        if lang and lang in TARGET_LANGS and lang not in seen:
            seen.add(lang)
            out.append(lang)

    # yêu cầu: chuẩn hóa chữ thường -> đã lowercase
    return ",".join(out)

def main():
    df = pd.read_csv(INPUT_PATH)

    if "skill" not in df.columns:
        raise ValueError(f"Không thấy cột 'skill' trong file. Các cột hiện có: {list(df.columns)}")

    df["skill"] = df["skill"].apply(clean_skill_cell)
    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print("✅ Done!")
    print("Input :", INPUT_PATH)
    print("Output:", OUTPUT_PATH)
    print(df.head(3))

if __name__ == "__main__":
    main()


✅ Done!
Input : job_2025_step1_clean.csv
Output: job_2025_step2_skill_clean.csv
                           title skill  \
0       AUTOMATION TEST ENGINEER         
1  Information Security Engineer         
2   Business Solutions Architect         

                                             summary  year  
0  Looking for Selenium engineers...must have sol...  2025  
1  The University of Chicago has a rapidly growin...  2025  
2  GalaxE.SolutionsEvery day, our solutions affec...  2025  


XEM SỐ DÒNG SKILL BỊ NULL và loại bỏ

In [4]:
import pandas as pd

# ====== CONFIG ======
INPUT_PATH = "job_2025_step2_skill_clean.csv"
OUTPUT_PATH = "job_2025_step3_drop_empty_skill.csv"

def main():
    df = pd.read_csv(INPUT_PATH)

    if "skill" not in df.columns:
        raise ValueError(f"Không thấy cột 'skill'. Các cột hiện có: {list(df.columns)}")

    total_rows = len(df)

    # xác định skill null / rỗng
    mask_empty_skill = (
        df["skill"].isna() |
        (df["skill"].astype(str).str.strip() == "")
    )

    empty_skill_count = mask_empty_skill.sum()

    print(f"📊 Tổng số dòng           : {total_rows}")
    print(f"⚠️  Số dòng skill null/rỗng: {empty_skill_count}")

    # loại bỏ các dòng skill rỗng
    df_clean = df.loc[~mask_empty_skill].reset_index(drop=True)

    print(f"✅ Số dòng sau khi clean  : {len(df_clean)}")

    # lưu file mới
    df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print("📁 File output:", OUTPUT_PATH)
    print(df_clean.head(3))

if __name__ == "__main__":
    main()


📊 Tổng số dòng           : 22000
⚠️  Số dòng skill null/rỗng: 13794
✅ Số dòng sau khi clean  : 8206
📁 File output: job_2025_step3_drop_empty_skill.csv
                                               title skill  \
0                                    DevOps Engineer    go   
1                                   Network Engineer    go   
2  Sr. Web Application Developer (Cloud Team) - C...    c#   

                                             summary  year  
0  Midtown based high tech firm has an immediate ...  2025  
1  Network Engineer Job Description A Network Eng...  2025  
2  Bluebeam is looking for talented sr. web devel...  2025  


tiếp tục tới việc clean titile và summary 
1. Xóa HTML tags (<br>,<p>...) 
2. Xóa URL rác 
3. Xóa khoảng trắng thừa 
4. xóa các từ vô nghĩa (and, or, show more, show less....)
5. Đưa về chữ thường

In [7]:
import re
import html
import pandas as pd

INPUT_PATH = "job_2025_step3_drop_empty_skill.csv"
OUTPUT_PATH = "job_2025_step4_clean_title_summary.csv"

# 1) HTML tags
HTML_TAG_RE = re.compile(r"<[^>]+>")

# 2) URL rác
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)

# 4) stop-phrases / từ vô nghĩa
STOP_PHRASES = [
    "and", "or",
    "show more", "show less",
    "read more", "read less",
    "see more", "see less",
    "learn more",
    "click here",
]

STOP_RE = re.compile(
    r"\b(?:%s)\b" % "|".join(map(re.escape, STOP_PHRASES)),
    re.IGNORECASE
)

# 3) khoảng trắng thừa
WS_RE = re.compile(r"\s+")

def clean_text(s: str) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""

    s = str(s)

    # decode HTML entities (&amp; -> &)
    s = html.unescape(s)

    # remove URLs
    s = URL_RE.sub(" ", s)

    # remove HTML tags
    s = HTML_TAG_RE.sub(" ", s)

    # remove stop-phrases
    s = STOP_RE.sub(" ", s)

    # normalize whitespace
    s = WS_RE.sub(" ", s).strip()

    # ====== NEW: lowercase toàn bộ ======
    s = s.lower()

    return s

def main():
    df = pd.read_csv(INPUT_PATH)

    for col in ["title", "summary"]:
        if col not in df.columns:
            raise ValueError(f"Không thấy cột '{col}'. Hiện có: {list(df.columns)}")

    df["title"] = df["title"].apply(clean_text)
    df["summary"] = df["summary"].apply(clean_text)

    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print("✅ Step 4 done (lowercase added)")
    print("📁 Output:", OUTPUT_PATH)
    print(df[["title", "summary"]].head(3))

if __name__ == "__main__":
    main()


✅ Step 4 done (lowercase added)
📁 Output: job_2025_step4_clean_title_summary.csv
                                               title  \
0                                    devops engineer   
1                                   network engineer   
2  sr. web application developer (cloud team) - c...   

                                             summary  
0  midtown based high tech firm has an immediate ...  
1  network engineer job description a network eng...  
2  bluebeam is looking for talented sr. web devel...  


ĐỔI SANG ĐÚNG ĐỊNH DẠNG DATASET VÀ THÔNG KÊ DỮ LIỆU

In [8]:
import pandas as pd

# ====== CONFIG ======
INPUT_PATH = "job_2025_step4_clean_title_summary.csv"
OUTPUT_PATH = "job_2025_step5_transformed.csv"

def main():
    df = pd.read_csv(INPUT_PATH)

    required_cols = ["title", "summary", "skill", "year"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Thiếu cột '{col}'. Hiện có: {list(df.columns)}")

    # ====== TRANSFORM ======
    new_df = pd.DataFrame({
        "raw_text": (
            df["title"].fillna("").astype(str).str.strip()
            + " "
            + df["summary"].fillna("").astype(str).str.strip()
        ).str.strip(),
        "language": df["skill"],
        "event_date": df["year"]
    })

    # ====== STATS ======
    total_rows = len(new_df)
    null_stats = new_df.isna().sum()

    print("📊 THỐNG KÊ DATA")
    print(f"👉 Tổng số dòng: {total_rows}")
    print("👉 Số null mỗi cột:")
    print(null_stats)

    # ====== SAVE ======
    new_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print("\n✅ Done!")
    print("📁 File output:", OUTPUT_PATH)
    print(new_df.head(3))

if __name__ == "__main__":
    main()


📊 THỐNG KÊ DATA
👉 Tổng số dòng: 8206
👉 Số null mỗi cột:
raw_text      0
language      0
event_date    0
dtype: int64

✅ Done!
📁 File output: job_2025_step5_transformed.csv
                                            raw_text language  event_date
0  devops engineer midtown based high tech firm h...       go        2025
1  network engineer network engineer job descript...       go        2025
2  sr. web application developer (cloud team) - c...       c#        2025
